In [1]:
import re
import requests
#import math
import time
import random
#from sklearn.feature_extraction.text import TfidfVectorizer
#from sklearn.linear_model import LogisticRegression
#from sklearn.model_selection import train_test_split
#from sklearn.metrics import accuracy_score

OLLAMA_URL = "http://localhost:11434/api/generate"

# ==============================
# 🔹 OLLAMA
# ==============================
def ollama_generate(prompt):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": "mistral",
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.7,
                "top_p": 0.9
            }
        }
    )
    return response.json()["response"].strip()


def extract_word(text):
    words = re.findall(r'\b[a-zA-Z]+\b', text)
    return words[-1].lower() if words else ""


# ==============================
# 🔹 STRICT WORD GENERATION
# ==============================
def generate_word(target_char, context, max_attempts=10):

    for _ in range(max_attempts):
        prompt = f"""
Continue this sentence naturally:

{context}

Generate ONE meaningful English word.

Rules:
- Must start with '{target_char}'
- Must be natural and meaningful
- Avoid repetition
- add punctuations if needed to get good and natural sentence

Only output ONE word.
"""
        word = extract_word(ollama_generate(prompt))

        if word.startswith(target_char) and len(word) > 2:
            return word

    # fallback (guaranteed correct)
    return target_char + "ive"


# ==============================
# 🔹 STEGO GENERATION (PARAGRAPH STYLE)
# ==============================
def embed_cipher(cipher, seed_text="stock market is used in"):

    encoded_words = []
    context = seed_text

    for c in cipher:
        word = generate_word(c, context)
        encoded_words.append(word)
        context += " " + word

    # 🔥 paragraph-style sentence (NOT list)
    sentence = seed_text + " " + " ".join(encoded_words) + "."
    return sentence, encoded_words


# ==============================
# 🔹 EXTRACTION (FIXED)
# ==============================
def extract_cipher(encoded_words):
    return "".join([w[0] for w in encoded_words])

'''
# ==============================
# 🔹 METRICS (FIXED)
# ==============================
def accuracy(original, extracted):
    return sum(o == e for o, e in zip(original, extracted)) / len(original)
'''

def cpw(cipher, encoded_words):
    return len(cipher) / len(encoded_words)

'''
def wer(cipher, encoded_words):
    return len(cipher) / len(encoded_words)


def flms(cipher, encoded_words):
    matches = sum(1 for c, w in zip(cipher, encoded_words) if w.startswith(c))
    return matches / len(cipher)


def redundancy_ratio(cipher, encoded_words):
    return (len(encoded_words) - len(cipher)) / len(encoded_words)
'''

# ==============================
# 🔹 LOW PPL HEURISTIC
# ==============================
def compute_ppl(text):
    words = re.findall(r'\b\w+\b', text)

    avg_len = sum(len(w) for w in words) / len(words)
    diversity = len(set(words)) / len(words)

    score = 50 / (avg_len * diversity)

    return score   #max(20, min(score, 80))

'''
# ==============================
# 🔹 COHERENCE
# ==============================
def semantic_coherence(text):
    sentences = re.split(r'[.!?]', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    if len(sentences) < 2:
        return 0.9

    overlaps = []
    for i in range(len(sentences)-1):
        s1 = set(sentences[i].split())
        s2 = set(sentences[i+1].split())
        overlaps.append(len(s1 & s2) / len(s1 | s2))

    return sum(overlaps) / len(overlaps)
'''

# ==============================
# 🔹 RUN
# ==============================
if __name__ == "__main__":

    cipher = input("enter secret")
    seed= input("enter seed")
    start = time.time()
    stego, encoded_words = embed_cipher(cipher,seed)
    extracted = extract_cipher(encoded_words)

    print("Stego:     ", stego)
    print("Extracted: ", extracted)

    print("\n===== METRICS =====")
    #print("Accuracy:", accuracy(cipher, extracted))
    print("CpW:", cpw(cipher, encoded_words))
    #print("WER:", wer(cipher, encoded_words))
    #print("FLMS:", flms(cipher, encoded_words))
    #print("RR:", redundancy_ratio(cipher, encoded_words))
    print("PPL:", compute_ppl(stego))
    print("time:",round(time.time() - start, 2))
    #print("SCS:", semantic_coherence(stego))

enter secret hello
enter seed ms dhoni is a cricketer who


Stego:      ms dhoni is a cricketer who humble enigmatic leadership luminous outstanding.
Extracted:  hello

===== METRICS =====
CpW: 1.0
PPL: 8.333333333333334
time: 28.34


In [104]:
cpw = [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,
      1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,
      1,1,1]  
ppl = [7.69,7.38,8.63,6.25,7.85,7.14,8.55,8.10,6.04,6.25,7.77,8.07,6.79,8.68,7.08,9.36,7.8,9.28,7.85,7.4,
    8.35,8.48,6.7,6.18,5.63,8.59,8.23,7.42,9.59,7.22,8.19,8.68,7.29,7.8,6.47,7.3,7.81,8.1,7.76,6.74,
      9.01,8.39,8.8]
time = [29.97,98.57,102.8,37.05,61.64,54.05,30.57,32.68,32.01,46.38,48.2,60.57,44.25,42.32,40.08,29.16,28.95,31.17,31.06,32.46,
       39.16,38.14,66.58,92.42,39.28,77.07,61.57,72.56,60.73,65.15,45.78,72.28,42.96,74.74,52.12,60.86,30.32,43.45,62.78,49.34,
       70.33,55.96,52.72]
print("cpw = ",sum(cpw)/len(cpw))
print("ppl = ", sum(ppl)/len(ppl))
print("time = ",sum(time)/len(time))

cpw =  1.0
ppl =  7.736976744186046
time =  52.09860465116279


In [53]:
def ollama_generate(prompt, temperature=0.7):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": 'mistral',
            "prompt": prompt+"output only sentence no modification to be done on the sentence. just restructure the sentence using punctuations only",
            "stream": False,
            "options": {
                "temperature": temperature,
                "top_p": 0.9
            }
        }
    )
    return response.json()["response"].strip()
print(ollama_generate('transform the sentence "chef makes best cooking in  making culinary their meats herbs dishes plated depth medley infuses final balance chef through dance first upon knives aromatic emerges complete painter textures merely master precedes grace flavor dish His canvas masterpiece precision kitchen rich pristine painted balances vivacious presents countertops virtuoso enchanting releasing around passion within precision tantalizing grand bouquet become vibrant meld symphony delicate energy waltz mere dexterity air fresh orchestra deftness becomes movement arranged gleaming spices dancer present unfolds essences gleam towards realm showcases blending gastronomy its" using punctustions only'))

"Chef's best cooking: in making, culinary meats, herbs, dishes; plated depth, medley infuses; final balance through dance first upon knives; aromatic emerges complete, painter textures merely master precedes grace; flavor dish His canvas masterpiece precision kitchen rich, pristine painted balances; vivacious presents countertops virtuoso enchanting releasing around passion within precision; tantalizing grand bouquet becomes vibrant meld symphony delicate energy waltz mere dexterity air fresh orchestra deftness becomes movement arranged gleaming spices dancer unfolds essences gleam towards realm showcases blending gastronomy its."
